In [1]:
!pip -q install transformers torch sentencepiece accelerate

In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")

def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").input_ids
    outputs = model.generate(inputs, max_new_tokens=512)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [5]:
documents = """
Document 1:
Node.js uses Google's V8 engine.

Document 2:
Express.js is a Node.js framework.

Document 3:
MongoDB is a NoSQL database.
"""

question = "Which framework is commonly used with Node.js?"

prompt = f"""
You are a company knowledge assistant.

CONTEXT:
{documents}

RULES:
1. Answer ONLY from context.
2. If answer not found say:
   'Information not available in provided context.'
3. Mention source document.

QUESTION:
{question}
"""

print(generate(prompt))


Express.js


In [6]:
from collections import Counter
import re

question = """
A company sells a product for ₹500.

Manufacturing Cost = ₹300
Marketing Cost = ₹50
Tax = ₹25

Calculate the profit per product using the following steps:
1. Calculate Total Cost: Total Cost = Manufacturing Cost + Marketing Cost + Tax
2. Calculate Profit: Profit = Selling Price - Total Cost

What is the profit per product?

Think step by step and give final answer as:
FINAL ANSWER: <value>
"""

results = []

for i in range(5):
    response = generate(question)

    print(f"\n--- Run {i+1} ---")
    print(response)

    # Updated regex to match 'The answer:' or 'FINAL ANSWER:'
    match = re.search(r'(?:FINAL ANSWER|The answer)[: ]*(.*)', response, re.IGNORECASE)

    if match:
        results.append(match.group(1).strip())

vote = Counter(results)

print("\n======================")
print("Majority Decision")
print("======================")

# Added a check to prevent IndexError if no results are found
if vote:
    print(vote.most_common(1)[0])
else:
    print("No valid answers were extracted from the model's responses.")



--- Run 1 ---
Total cost = 300 + 50 + 25 = 500. Selling price = 500 - 500 = 250. Profit = 250 - 500 = 250. The final answer: 250.

--- Run 2 ---
Total cost = 300 + 50 + 25 = 500. Selling price = 500 - 500 = 250. Profit = 250 - 500 = 250. The final answer: 250.

--- Run 3 ---
Total cost = 300 + 50 + 25 = 500. Selling price = 500 - 500 = 250. Profit = 250 - 500 = 250. The final answer: 250.

--- Run 4 ---
Total cost = 300 + 50 + 25 = 500. Selling price = 500 - 500 = 250. Profit = 250 - 500 = 250. The final answer: 250.

--- Run 5 ---
Total cost = 300 + 50 + 25 = 500. Selling price = 500 - 500 = 250. Profit = 250 - 500 = 250. The final answer: 250.

Majority Decision
('250.', 5)
